# Skip edges

`parse_layered()` layers a DAG. An original edge whose endpoints
are more than one layer apart is a **skip**, and in `kpnn2` a skip
is nothing special to compute: it is a column of the target
layer's hop mask, exactly like an edge from the layer directly
below.

`LayeredSpec.hops` holds one `Hop` per layer after the first, and
`hops[i].mask` carries **every** edge entering layer `i + 1`. A
hop whose target has parents further back reads several layers, so
its mask columns are those layers concatenated;
`gather_hop_inputs()` builds that input tensor for you.
`spec.skips` still lists which edges span layers, but only as
metadata for reporting.

The [Getting started](../getting-started/) notebook uses a graph
without skip edges. This page shows what changes when a graph has
them, which is less than you might expect.

## What a skip edge means

An adjacent chain such as `H1 -> H2` is one hop: a `MaskedLinear`
maps the layers feeding `H2` onto `H2`.

A skip such as `A -> H2` is still a directed edge into `H2`, so
`A` is an extra **parent of the target unit**. The skip crosses
every layer between source and target without touching them: the
source is not copied through dummy units, and `kpnn2` never
inserts identity neurons or generated node names.

![Hop masks](../figures/hop_masks.svg)

**Figure 1.** Every edge entering a layer is in that layer's hop
mask. `hops[1]` therefore reads layers 0 and 1, and `hops[2]`
reads layers 0, 1 and 2. Solid and dashed edges are handled by the
same matrix multiply; the dashes only mark which edges jump a
layer.

Three properties follow, and they are the reason the design looks
like this:

- **No edge can be dropped silently.** Applying a hop applies
  every parent of its target at once. There is no second call to
  remember, and `gather_hop_inputs()` raises `Kpnn2Error` if a
  layer that a hop reads was never stored.
- **The fan-in is right.** `MaskedLinear` initializes each output
  row from that row's mask degree. Because skip parents sit in the
  same row, a unit with two adjacent and three skip parents is
  initialized with fan-in five, not two.
- **A skip weight is an ordinary weight.** One weight per incoming
  edge, all in one matrix, all drawn from the same degree-aware
  initialization. There is no skip bias: one bias per unit stays
  on `MaskedLinear`.

In symbols, for target `H2` with adjacent parent `H1` and skip
parent `A`:

```text
H2 = relu( w_{H1→H2} H1 + w_{A→H2} A + b_{H2} )
```

`map_node_attributions()` labels original `layer_nodes` names, so
if a skip affects the prediction, that effect appears on the named
target unit and never on a dummy channel.

## A three-hop module

The example below is the graph in Figure 1: inputs `A` and `B`,
hidden units `H1` and `H2`, output `C`. Adjacent edges are
`A -> H1`, `B -> H1`, `H1 -> H2`, and `H2 -> C`. Skips are
`A -> H2`, `H1 -> C`, and `A -> C`.

In [1]:
import pandas as pd

import kpnn2 as k2

edgelist = pd.DataFrame(
    {
        "source": ["A", "B", "H1", "H2", "A", "H1", "A"],
        "target": ["H1", "H1", "H2", "C", "H2", "C", "C"],
    }
)
spec = k2.parse_layered(edgelist)
spec.layer_nodes, spec.skips

((('A', 'B'), ('H1',), ('H2',), ('C',)),
 (Skip(source='A', target='H2', source_layer=0, target_layer=2, source_index=0, target_index=0),
  Skip(source='H1', target='C', source_layer=1, target_layer=3, source_index=0, target_index=0),
  Skip(source='A', target='C', source_layer=0, target_layer=3, source_index=0, target_index=0)))

In [2]:
summary = [
    {
        "target_layer": hop.target_layer,
        "source_layers": hop.source_layers,
        "source_nodes": hop.source_nodes,
        "column_offsets": hop.column_offsets,
        "shape": tuple(hop.mask.shape),
        "ones": int(hop.mask.sum().item()),
    }
    for hop in spec.hops
]
n_ones = sum(entry["ones"] for entry in summary)
pd.DataFrame(summary), n_ones, len(edgelist)

(   target_layer source_layers    source_nodes column_offsets   shape  ones
 0             1          (0,)          (A, B)           (0,)  (1, 2)     2
 1             2        (0, 1)      (A, B, H1)         (0, 2)  (1, 3)     2
 2             3     (0, 1, 2)  (A, B, H1, H2)      (0, 2, 3)  (1, 4)     3,
 7,
 7)

`hops[0]` reads layer 0 alone, because layer 1 can only have
layer-0 parents. `hops[1]` also reads layer 0, since `A -> H2`
jumps a layer, and `hops[2]` reads all three earlier layers.

The mask columns follow `source_nodes`, which is the source layers
concatenated in ascending order. `column_offsets` gives the first
column of each source layer, so you can find any single edge:

- rows are `spec.layer_nodes[hop.target_layer]`
- columns are `hop.source_nodes`

The ones across all hop masks add up to the number of rows in the
edgelist. That equality is the guarantee: every edge of the prior
knowledge is in exactly one mask, so there is no way to build the
model and lose one.

One `MaskedLinear` per hop, and a `saved` dict mapping layer index
to that layer's finished tensor. `gather_hop_inputs()` picks the
layers each hop reads and concatenates them; when a hop reads one
layer it returns that tensor unchanged, with no copy.

The last hop stays linear here, which is a modeling choice. You
own the nonlinearities, the call order, and what goes into
`saved`.

In [3]:
import torch
import torch.nn.functional as F
from torch import nn


class Net(nn.Module):
    def __init__(
        self,
        spec: k2.LayeredSpec,
    ):
        super().__init__()
        self.spec = spec
        self.hops = nn.ModuleList(
            [
                k2.MaskedLinear(
                    hop.mask,
                    bias=False,
                )
                for hop in spec.hops
            ]
        )

    def forward(
        self,
        x,
    ):
        saved = {0: x}
        last = len(self.hops) - 1
        hidden = x
        for index, hop in enumerate(self.spec.hops):
            sources = k2.gather_hop_inputs(
                saved,
                hop,
            )
            hidden = self.hops[index](sources)
            if index < last:
                hidden = F.relu(hidden)
            saved[hop.target_layer] = hidden
        self.saved = saved
        return hidden


model = Net(spec)
model

Net(
  (hops): ModuleList(
    (0): ParametrizedMaskedLinear(
      in_features=2, out_features=1, bias=False
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _MaskParametrization()
        )
      )
    )
    (1): ParametrizedMaskedLinear(
      in_features=3, out_features=1, bias=False
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _MaskParametrization()
        )
      )
    )
    (2): ParametrizedMaskedLinear(
      in_features=4, out_features=1, bias=False
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _MaskParametrization()
        )
      )
    )
  )
)

### Numerical check

Every edge now has one weight in one matrix, so the whole model can
be pinned from a single dictionary of edge weights: adjacent edges
at `1`, and `w_{A→H2} = 0.3`, `w_{H1→C} = 0.2`, `w_{A→C} = 0.1`.
Bias is disabled. For input `A = 2`, `B = 0`:

```text
H1 = relu(A + B) = 2
H2 = relu(H1 + 0.3 A) = relu(2.6) = 2.6
C  = H2 + 0.2 H1 + 0.1 A = 3.2
```

For `A = -1`, `B = 0`, ReLU zeros the path through `H1` and `H2`,
but the skip `A -> C` still contributes:

```text
H1 = relu(-1) = 0
H2 = relu(0 + 0.3 (-1)) = 0
C  = 0 + 0.2 * 0 + 0.1 * (-1) = -0.1
```

Assigning `layer.weight = pinned` writes through the mask
parametrization into the trainable tensor; the mask then hides
whatever the pinned matrix says about absent edges.

In [4]:
edge_weights = {
    ("A", "H1"): 1.0,
    ("B", "H1"): 1.0,
    ("H1", "H2"): 1.0,
    ("A", "H2"): 0.3,
    ("H2", "C"): 1.0,
    ("H1", "C"): 0.2,
    ("A", "C"): 0.1,
}

with torch.no_grad():
    for layer, hop in zip(
        model.hops,
        spec.hops,
        strict=True,
    ):
        pinned = torch.zeros_like(hop.mask)
        targets = spec.layer_nodes[hop.target_layer]
        for column, source in enumerate(hop.source_nodes):
            for row, target in enumerate(targets):
                value = edge_weights.get((source, target))
                if value is not None:
                    pinned[row, column] = value
        layer.weight = pinned


def layer_values(x):
    with torch.no_grad():
        y = model(x)
        return {
            "H1": model.saved[1],
            "H2": model.saved[2],
            "C": y,
        }


x_pos = torch.tensor(
    [[2.0, 0.0]],
)
x_neg = torch.tensor(
    [[-1.0, 0.0]],
)
layer_values(x_pos), layer_values(x_neg)

({'H1': tensor([[2.]]), 'H2': tensor([[2.6000]]), 'C': tensor([[3.2000]])},
 {'H1': tensor([[0.]]), 'H2': tensor([[0.]]), 'C': tensor([[-0.1000]])})

## Step by step

`x` has one row per sample and columns `(A, B)` in
`spec.input_nodes` order.

**Save layer 0.** `saved = {0: x}` keeps the input tensor. The
hops into `H2` and `C` read the `A` column from this entry, and
`A` is never overwritten.

**Hop into layer 1.** `gather_hop_inputs(saved, hops[0])` returns
`x` itself, since that hop reads only layer 0. The
`MaskedLinear` maps `(A, B)` onto `H1`. Then ReLU, then
`saved[1] = h`.

**Hop into layer 2.** This hop reads layers 0 and 1, so the
gather concatenates `x` and `H1` into a three-column tensor
`(A, B, H1)`. One matrix multiply applies `w_{A→H2}` and
`w_{H1→H2}` together, and the `B` column is masked to zero. ReLU
then applies to the sum, so the skip term is inside `H2`'s
nonlinearity.

**Hop into layer 3.** This hop reads layers 0, 1 and 2, so the
gather builds `(A, B, H1, H2)`. The single multiply applies
`w_{A→C}`, `w_{H1→C}` and `w_{H2→C}`. No ReLU follows in this
listing, so those terms are not passed through a further
nonlinearity.

Nothing in this walkthrough is specific to skips. The same three
lines run for every layer, which is why a forgotten skip edge is
not a failure mode any more.

## Two things worth checking

The fan-in that `MaskedLinear` initializes from is the row degree
of the hop mask, so it counts skip parents. And a hop that reads a
layer you never stored is an error, not a silent omission.

In [5]:
fan_in = {
    name: int(degree)
    for hop in spec.hops
    for name, degree in zip(
        spec.layer_nodes[hop.target_layer],
        hop.mask.sum(dim=1).tolist(),
        strict=True,
    )
}

try:
    k2.gather_hop_inputs(
        {0: x_pos},
        spec.hops[2],
    )
    message = "no error"
except k2.Kpnn2Error as error:
    message = str(error)

fan_in, message

({'H1': 2, 'H2': 2, 'C': 3},
 'saved is missing layer 1. The hop into layer 3 reads layers [0, 1, 2].')